# 4. Agent conversationnel

L'agent répond à vos questions en interrogeant les données avec des **outils** :
contrôles automatiques, recherche d'exigences, lignes AMDEC, matrice de traçabilité.

Il ne modifie rien : ses suggestions sont enregistrées comme **propositions à valider**.

Nécessite un modèle qui gère l'appel d'outils (Qwen 2.5+, Llama 3.1+, Mistral Small…).

In [1]:
# Rend le package safety_agent importable depuis le dossier notebooks/
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
EXAMPLES = ROOT / "examples"

import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
# Configuration du modèle hébergé (Ollama par défaut)
BASE_URL = "http://localhost:11434/v1"   # vLLM : "http://serveur:8000/v1"
MODEL = "qwen2.5:7b"     # qwen2.5:14b ou plus si vous avez une carte graphique

from safety_agent.notebook import get_llm
llm = get_llm(BASE_URL, MODEL)   # bascule en MODE DÉMO si le serveur ne répond pas

⚠️  Serveur LLM injoignable à http://localhost:11434/v1 (<urlopen error [Errno 111] Connection refused>)
➡️  MODE DÉMO : réponses simulées, sans IA. Lancez Ollama/vLLM pour de vraies analyses.


In [3]:
from safety_agent.loaders import load_project

# Remplacez par vos fichiers (.csv ou .xlsx)
REQS = EXAMPLES / "bms_exigences.csv"
FMEA = EXAMPLES / "bms_amdec.csv"

project, load_findings = load_project(REQS, FMEA)
print(f"{len(project.requirements)} exigences, {len(project.fmea)} lignes AMDEC")

11 exigences, 6 lignes AMDEC


In [4]:
from safety_agent.agent import SafetyAgent

agent = SafetyAgent(llm, project, on_tool=lambda name, args: print(f"  🔧 {name} {args or ''}"))

In [5]:
print(agent.ask("Fais un état des lieux : quelles sont les erreurs bloquantes ?"))

  🔧 run_rule_checks 
[DÉMO — réponse simulée, pas d'IA] 21 anomalies détectées, dont 9 erreurs :
- FM-06 : Cotation O manquante.
- FM-01 : Priorité d'action H sans action définie (action ou justification obligatoire).
- FM-04 : Priorité d'action H sans action définie (action ou justification obligatoire).
- FM-04 : Exigence(s) liée(s) inconnue(s) : FSR-77.
- FM-04 : Effet critique pour la sécurité (S≥9) sans exigence de sécurité tracée.
- FM-06 : Effet critique pour la sécurité (S≥9) sans exigence de sécurité tracée.
- SW-01 : Exigence orpheline : aucun parent (rupture de traçabilité vers un objectif de sécurité).
- TSR-02 : Parent inconnu : FSR-99.
- FSR-05 : ASIL A inférieur à l'ASIL du parent SG-02 (ASIL B) sans décomposition déclarée.


In [6]:
# Posez vos propres questions (l'agent garde le contexte de la conversation)
print(agent.ask("Quels modes de défaillance de gravité 9 ou 10 n'ont pas d'action ? Propose une action pour chacun."))

  🔧 run_rule_checks 
  🔧 record_proposal {'kind': 'action', 'target': 'FM-01', 'content': "[DÉMO] Définir une action de réduction du risque (diagnostic supplémentaire ou redondance) ou justifier formellement l'absence d'action.", 'rationale': "FM-01 : priorité d'action H sans action (FMEA-010)."}
  🔧 record_proposal {'kind': 'action', 'target': 'FM-04', 'content': "[DÉMO] Définir une action de réduction du risque (diagnostic supplémentaire ou redondance) ou justifier formellement l'absence d'action.", 'rationale': "FM-04 : priorité d'action H sans action (FMEA-010)."}
[DÉMO — réponse simulée, pas d'IA] 2 proposition(s) d'action enregistrée(s) pour les lignes en priorité H sans action. Voir le tableau des propositions.


## Propositions enregistrées par l'agent

In [7]:
from safety_agent.notebook import proposals_df
proposals_df(agent.proposals)

,Type,Cible,Proposition,Justification,Statut
0,action,FM-01,[DÉMO] Définir une action de réduction du risque (diagnostic supplémentaire ou redondance) ou justifier formellement...,FM-01 : priorité d'action H sans action (FMEA-010).,À VALIDER
1,action,FM-04,[DÉMO] Définir une action de réduction du risque (diagnostic supplémentaire ou redondance) ou justifier formellement...,FM-04 : priorité d'action H sans action (FMEA-010).,À VALIDER
